# 실습 9: Ollama LLM을 활용한 HTTP 분류 (2교시)

특성 추출 없이 **HTTP 요청 텍스트를 그대로** Ollama gemma3:1b에 보여주고
정상/공격을 분류합니다.

**사전 조건**:
- Ollama 서버 실행 중 (`ollama serve`)
- `gemma3:1b` 모델 다운로드됨 (`ollama list`로 확인)
- 7주차 `processed_data.pkl` 존재 (LLM용 텍스트 샘플 포함)


In [1]:
# %% [Setup] 패키지 import 및 LLM용 샘플 로드
import pickle
import time
import json
import re
import pandas as pd
from urllib.parse import unquote
from sklearn.metrics import accuracy_score, f1_score, classification_report

import ollama  # pip install ollama

with open("../7주차/processed_data.pkl", "rb") as f:
    data = pickle.load(f)

llm_sample = data["llm_sample"].head(100).reset_index(drop=True)
print(f"분류 대상: {len(llm_sample)}건")
print(f"라벨 분포: 정상 {(llm_sample.get('is_attack',0)==0).sum()}건 / "
      f"공격 {(llm_sample.get('is_attack',0)==1).sum()}건")


분류 대상: 100건
라벨 분포: 정상 56건 / 공격 44건


c:\Users\asus\anaconda3\envs\dmu\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## 1. 분류 프롬프트 설계

LLM 응답을 안정적으로 파싱하기 위해:
- **Few-shot 예시** 2개로 출력 형식을 학습시킴
- **JSON 형태**로 응답하도록 강제
- 영어 프롬프트(gemma3:1b가 영어에 더 정확)

In [ ]:
# %% [1] HTTP 텍스트 재구성 + 프롬프트 함수
def build_http_text(row) -> str:
    method = row.get("method", "GET")
    url    = unquote(str(row.get("url", "")), encoding="latin-1")
    body   = str(row.get("body_decoded", row.get("body", "")) or "")
    text   = f"{method} {url} HTTP/1.1"
    if body and body != "nan":
        text += f"\nBody: {body[:200]}"
    return text


PROMPT_TEMPLATE = 'You are a web security expert. Classify each HTTP request as "Normal" or "Anomalous" and provide a brief reason.\n\nExamples:\nRequest: GET /index.jsp HTTP/1.1\nOutput: {{"label": "Normal", "reason": "Standard page request, no suspicious pattern"}}\n\nRequest: GET /search?q=\' OR \'1\'=\'1 HTTP/1.1\nOutput: {{"label": "Anomalous", "reason": "Classic SQL Injection pattern with OR 1=1"}}\n\nNow classify:\nRequest: {http_text}\nOutput:'


def classify_with_llm(http_text: str, model: str = "gemma3:1b") -> dict:
    """Ollama로 HTTP 요청 분류 -> {label, reason}"""
    prompt = PROMPT_TEMPLATE.format(http_text=http_text)
    response = ollama.chat(
        model=model,
        messages=[{"role":"user","content":prompt}],
        options={"temperature": 0},  # 결정성 높이기
    )
    text = response["message"]["content"]

    # JSON 추출 - LLM이 가끔 앞뒤 설명을 붙임
    match = re.search(r"\{[^{}]*\}", text, re.DOTALL)
    if not match:
        return {"label":"Unknown", "reason": text[:80]}
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return {"label":"Unknown", "reason": text[:80]}


# 단건 테스트
test_text = "GET /tienda1/publico/anadir.jsp?id=2'+OR+'1'='1 HTTP/1.1"
print("입력:", test_text)
print("응답:", classify_with_llm(test_text))


<>:12: SyntaxWarning: invalid escape sequence '\,'
<>:12: SyntaxWarning: invalid escape sequence '\,'
C:\Users\asus\AppData\Local\Temp\ipykernel_4172\2723904054.py:12: SyntaxWarning: invalid escape sequence '\,'
  PROMPT_TEMPLATE = '''You are an HTTP request classifier. Analyze requests and output JSON only.


입력: GET /tienda1/publico/anadir.jsp?id=2'+OR+'1'='1 HTTP/1.1
응답: {'label': 'Anomalous', 'reason': 'SQL injection OR 1=1 pattern'}


## 2. 100건 분류 + 시간 측정

CPU 환경 기준 건당 1~3초가 표준. 100건 ≈ 2~5분 소요.

In [12]:
# %% [2] 100건 분류
results = []
start = time.time()

for i, row in llm_sample.iterrows():
    http_text = build_http_text(row)
    result = classify_with_llm(http_text)
    true_label = "Anomalous" if row.get("is_attack", 0) == 1 else "Normal"
    results.append({
        "idx": i,
        "true": true_label,
        "pred": result.get("label", "Unknown"),
        "reason": result.get("reason", "")[:120],
        "http_short": http_text[:100],
    })
    if (i + 1) % 10 == 0:
        elapsed = time.time() - start
        print(f"  {i+1}/{len(llm_sample)}건 완료 "
              f"({elapsed:.1f}초, 건당 {elapsed/(i+1):.2f}초)")

llm_time = time.time() - start
llm_df = pd.DataFrame(results)
print(f"\n총 소요: {llm_time:.1f}초")
print(f"1만 건 환산: 약 {llm_time/100*10000/60:.0f}분")


  10/100건 완료 (56.3초, 건당 5.63초)
  20/100건 완료 (86.5초, 건당 4.33초)
  30/100건 완료 (121.8초, 건당 4.06초)
  40/100건 완료 (178.4초, 건당 4.46초)
  50/100건 완료 (223.1초, 건당 4.46초)
  60/100건 완료 (263.6초, 건당 4.39초)
  70/100건 완료 (295.0초, 건당 4.21초)
  80/100건 완료 (328.0초, 건당 4.10초)
  90/100건 완료 (381.1초, 건당 4.23초)
  100/100건 완료 (414.9초, 건당 4.15초)

총 소요: 414.9초
1만 건 환산: 약 692분


In [13]:
# %% [3] 정확도/F1 계산
llm_df["pred_clean"] = llm_df["pred"].replace({"Unknown":"Normal"})
y_true = (llm_df["true"] == "Anomalous").astype(int)
y_pred = (llm_df["pred_clean"] == "Anomalous").astype(int)

llm_acc = accuracy_score(y_true, y_pred)
llm_f1  = f1_score(y_true, y_pred)

print(f"LLM 정확도: {llm_acc:.4f}")
print(f"LLM F1:    {llm_f1:.4f}")
print(f"분류 실패(Unknown): {(llm_df['pred']=='Unknown').sum()}건")
print()
print(classification_report(y_true, y_pred, target_names=["Normal","Anomalous"]))


LLM 정확도: 0.6300
LLM F1:    0.3934
분류 실패(Unknown): 3건

              precision    recall  f1-score   support

      Normal       0.61      0.91      0.73        56
   Anomalous       0.71      0.27      0.39        44

    accuracy                           0.63       100
   macro avg       0.66      0.59      0.56       100
weighted avg       0.65      0.63      0.58       100



## 3. 자연어 판단 근거 검토 ★

LLM의 가장 큰 강점: **왜 그렇게 판단했는지** 사람이 읽을 수 있는 문장으로 설명.
이는 SOC(보안관제) 분석가가 1차 분류를 검토할 때 매우 유용합니다.

In [6]:
# %% [4] 공격으로 판단한 사례 + LLM 근거
print("=== LLM이 공격으로 판단한 사례 (상위 5건) ===\n")
attack_pred = llm_df[llm_df["pred"] == "Anomalous"].head(5)
for _, r in attack_pred.iterrows():
    correct = "OK" if r["true"] == "Anomalous" else "오탐"
    print(f"[{correct}] 실제={r['true']:10s}  요청: {r['http_short']}")
    print(f"   - LLM 근거: {r['reason']}\n")


=== LLM이 공격으로 판단한 사례 (상위 5건) ===

[OK] 실제=Anomalous   요청: GET /tienda1/miembros/editar.jsp?modo=registro&loginA=lieure&password=rEbatible&nombre=Tarciano&apel
   - LLM 근거: The request includes a complex and seemingly unnecessary set of parameters (dni, direccion, B, B1, etc.) that are not ty

[오탐] 실제=Normal      요청: POST /tienda1/publico/entrar.jsp HTTP/1.1
Body: errorMsg=Credenciales+incorrectas
   - LLM 근거: Sending sensitive data (credentials) in the request body. This is a common attack vector for credential stuffing.

[OK] 실제=Anomalous   요청: POST /tienda1/publico/vaciar.jsp HTTP/1.1
Body: B2=Vaciar+carrito%3CSCRIPT%3Ealert%28%22Paros%22%29%
   - LLM 근거: The request body contains a seemingly random B2=Vaciar+carrito%3BSCRIPT%3Ealert%28%22Paros%22%29, which is a common tact

[OK] 실제=Anomalous   요청: POST /tienda1/publico/vaciar.jsp HTTP/1.1
Body: B2A=Vaciar+carrito
   - LLM 근거: The request body contains a seemingly random string ('B2A=Vaciar+carrito') that doesn't appear to be part of

In [7]:
# %% [5] 결과 저장 (3교시에서도 활용)
with open("llm_classification_results.pkl", "wb") as f:
    pickle.dump({
        "llm_df": llm_df,
        "llm_acc": llm_acc,
        "llm_f1": llm_f1,
        "llm_time": llm_time,
        "n_samples": len(llm_sample),
    }, f)
print(">> llm_classification_results.pkl 저장 완료")


>> llm_classification_results.pkl 저장 완료


**다음**: `comparison_analysis.ipynb`로 1교시 ML 결과와 종합 비교합니다.